# 02 — Anti-UAV410 → a thermal EdgeTAM, end to end

One runtime, one **Run all**: download the dataset onto this machine, label it,
fine-tune EdgeTAM at 512×512, and measure whether it helped on sequences it
never trained on.

| | |
|---|---|
| **in** | Anti-UAV410 — 410 thermal sequences, 8.7 GB, straight off the authors' Drive |
| **out** | `checkpoints/edgetam_thermal_512.pt`, plus a stock-vs-fine-tuned score on `val` and `test` |
| **needs** | a CUDA GPU and ~25 GB of disk. Nothing on your own Drive is required |
| **takes** | ~30 min to fetch, ~40 min to label, ~1–2 h to train, at the defaults below |

The checkpoint drops into `configs/edgetam_512_thermal.yaml`, the ONNX export
and the TensorRT build **with nothing else changed** — the fine-tune moves
weights, never architecture.

---

## The dataset, and why it is this one

**Anti-UAV410**: 410 thermal-infrared sequences of drones in the wild, 438K
hand-annotated boxes, **640×512 8-bit mono**, split into `train` / `val` /
`test`. Each sequence is a folder of JPEGs plus one `IR_label.json`:

```
Anti-UAV410/
├── train/  <sequence>/  0.jpg 1.jpg … IR_label.json   {"exist": [1,1,0,…],
├── val/    <sequence>/  …                              "gt_rect": [[x,y,w,h], [], …]}
└── test/   <sequence>/  …
```

Three properties decide everything downstream:

- A **512×512 window is native pixels — no resize at all.** That is exactly the
  `crop512` input mode the Orin deployment already runs. Training on a resized
  view and deploying on a cropped one would train the wrong thing.
- **`exist` is annotated per frame.** Direct supervision for
  `object_score_logits` — the signal behind the failure this project hit, where
  one hard frame writes `no_obj_ptr` into the memory bank and the next seven
  frames read it back.
- **It is video.** The memory bank *is* the model; a set of stills cannot train
  or evaluate it.

`train` trains, `val` selects the checkpoint, `test` is touched exactly once at
the end and never influences anything.

## The one environment fact worth knowing

EdgeTAM installs itself **as the package `sam2`** — it is a fork. So Meta's
`sam2` must never be installed alongside it. The SAM 2.1 *teacher* here runs
through **`transformers`**, which carries an independent SAM2 implementation
under a different name, which is what lets labelling and training share one
runtime instead of two.

In [ ]:
# --- Runtime, repo, GPU -------------------------------------------------
import os, sys
from pathlib import Path

REPO   = Path("/content/sam-dedection")
BRANCH = "claude/fervent-fermat-yvfyd3"

if not REPO.exists():
    !git clone -q -b {BRANCH} https://github.com/yigitkayabagci/sam-dedection.git {REPO}
!git -C {REPO} fetch -q origin {BRANCH}
!git -C {REPO} checkout -q {BRANCH} && git -C {REPO} merge -q --ff-only origin/{BRANCH}
os.chdir(REPO)
sys.path.insert(0, str(REPO))

# Long clip-mode activations fragment the allocator badly; expandable segments
# is worth several batch sizes on a large card and costs nothing on a small one.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /content | tail -1

In [ ]:
# --- Dependencies -------------------------------------------------------
# EdgeTAM (as `sam2`) and transformers' own SAM2 coexist deliberately: the
# teacher goes through transformers, so no second runtime is needed.
!bash scripts/setup_edgetam.sh 2>&1 | tail -5
!pip install -q -r requirements.txt
!pip install -q "transformers>=4.56" gdown

import sam2, transformers
assert hasattr(transformers, "Sam2Model"), "transformers is too old for the SAM 2.1 teacher"
print(f"sam2 (EdgeTAM) {Path(sam2.__file__).parent}\ntransformers {transformers.__version__}")
assert Path("third_party/EdgeTAM/checkpoints/edgetam.pt").is_file(), \
    "edgetam.pt did not download -- rerun scripts/setup_edgetam.sh and read its output"

In [ ]:
# The contracts everything below depends on, tested with no GPU and no
# checkpoint. If these fail, nothing after this point is worth running.
!python -m unittest tests.test_clip_loop tests.test_training_losses \
    tests.test_antiuav_dataset tests.test_accuracy tests.test_pseudo_labels \
    tests.test_loader tests.test_fetch_antiuav410 2>&1 | tail -3

## Settings

The only cell you should need to edit. The defaults are sized for one Colab
session on a large card: a subset big enough for the fine-tune to mean
something, small enough to finish.

**The dataset goes on local disk, not on Drive.** Training reads a few hundred
thousand small JPEGs in random order, and the Drive FUSE mount serves those at
a few hundred files a second — an order of magnitude slower than the GPU
consumes them. Drive is used, optionally, for the one thing worth surviving the
runtime: the checkpoint.

`LABEL_STRIDE` is the honest lever on labelling time. A skipped frame keeps its
`exist` flag and its box, so it still trains the object-score head and still
contributes a box-projection term — it just gets no mask. At 25 fps,
consecutive masks are nearly identical anyway.

In [ ]:
DATA_DIR = Path("/content/data")      # the dataset -- local NVMe, never Drive
WORK     = Path("/content/work")      # labels + manifest, this runtime only
SPLITS   = ("train", "val", "test")   # drop "test" to save ~3 GB and a few minutes

SIZE                   = 512          # model input == the native crop size
CLIP_LEN, CLIP_STRIDE  = 8, 2         # 8 frames, every other one: 0.6 s of video
TRAIN_SEQUENCES        = 60           # of the whole train split; None = all
VAL_SEQUENCES          = 16
TEST_SEQUENCES         = 12           # scored once, at the very end

TEACHER_ID   = "facebook/sam2.1-hiera-large"   # -hiera-base-plus is ~2.5x faster
LABEL_STRIDE = 3                      # mask every 3rd annotated frame
ZOOM, MIN_CROP = 4.0, 128

STEPS_PER_EPOCH = 400                 # a "pass" over 60k overlapping clips is a day
VAL_BATCHES     = 24
BATCH_CEILING   = 128                 # auto_batch_size never probes past this
LOADER_WORKERS  = min(2 * (os.cpu_count() or 4), 24)   # clips read at once
PREFETCH_DEPTH  = 2                   # batches held in host RAM ahead of the GPU
SEED            = 0

LABELS = WORK / "labels"
CKPT   = REPO / "checkpoints"
for d in (DATA_DIR, WORK, LABELS, CKPT):
    d.mkdir(parents=True, exist_ok=True)

# The teacher is the one place raw VRAM buys throughput directly.
import torch
VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
TEACHER_BATCH = 8 if VRAM < 24 else 16 if VRAM < 48 else 32
print(f"{VRAM:.0f} GiB of VRAM -> teacher batch {TEACHER_BATCH}")
print(f"{os.cpu_count()} cores -> {LOADER_WORKERS} loader threads, "
      f"{PREFETCH_DEPTH} batches queued ahead")

# Optional, and only for the checkpoint: a Colab runtime is not storage.
MIRROR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    MIRROR = Path("/content/drive/MyDrive/edgetam-thermal")
    MIRROR.mkdir(parents=True, exist_ok=True)
    print(f"checkpoints will be mirrored to {MIRROR}")
except Exception as exc:
    print(f"no Drive ({type(exc).__name__}) -- everything stays local and dies "
          f"with the runtime")

## Getting the data

8.7 GB from the authors' Google Drive, unpacked in place. `tools/fetch_antiuav410.py`
does three things a `gdown | unzip` does not:

- **checks what it downloaded** — Drive answers a quota-exceeded request with an
  HTML page and an HTTP 200, which becomes a 3 KB "zip" that fails twenty
  minutes later with an unrelated error;
- **extracts in parallel** — 440K members, and zlib releases the GIL;
- **verifies the layout it produced**, rather than assuming the archive's
  top-level folder is spelled the way the README spells it.

It is safe to re-run: an already-extracted split is skipped. If Drive's daily
quota for the file is spent, add it to your own Drive from the [share
link](https://drive.google.com/file/d/1zsdazmKS3mHaEZWS2BnqbYHPEcIaH5WR/view),
mount it, and pass `--zip /content/drive/MyDrive/Anti-UAV410.zip`.

In [ ]:
!python tools/fetch_antiuav410.py --dest {DATA_DIR} --splits {" ".join(SPLITS)}

In [ ]:
from tools.fetch_antiuav410 import dataset_root, describe, find_splits

splits = find_splits(DATA_DIR)
missing = [s for s in SPLITS if s not in splits]
assert not missing, f"{missing} did not extract -- rerun the cell above"

DATA = dataset_root(splits)      # what --data and list_sequences() both want
print(describe(splits))
print(f"\nDATA = {DATA}")

In [ ]:
# --- What is actually in it --------------------------------------------
import numpy as np
from src.training import frame_shape, list_sequences

train = list_sequences(DATA, "train")[:TRAIN_SEQUENCES]
val   = list_sequences(DATA, "val")[:VAL_SEQUENCES]
height, width = frame_shape(train[0].frames[0])

for name, sequences in (("train", train), ("val", val)):
    frames  = sum(len(s) for s in sequences)
    visible = sum(int(s.labels.exist.sum()) for s in sequences)
    print(f"{name:<6} {len(sequences):>3} sequences  {frames:>7} frames  "
          f"{visible:>7} annotated ({visible / max(frames, 1):.1%})")

print(f"\nframe size {width}x{height}  ->  a {SIZE} crop is "
      f"{'native pixels, no resize' if min(width, height) >= SIZE else 'UPSCALED'}")
assert min(width, height) >= SIZE, \
    f"a {SIZE} window does not fit in a {width}x{height} frame; lower SIZE"

In [ ]:
# --- How big are the targets, and do clips stay on native pixels? -------
# The first decides whether notebook 05 (size-adaptive inference) is worth its
# latency. The second is a property of this dataset the training loop has to
# live with: a clip gets a fixed 512 window if the target's whole excursion
# fits in one, and otherwise falls back to resizing the entire frame. The
# window is fixed for the clip on purpose -- SAM 2's memory bank stores
# features in input coordinates, so a window that moved between frames would
# put every stored memory in a different frame of reference from its reader.
from src.training import sample_clips

sides = np.concatenate([
    np.nanmax(np.stack([s.labels.boxes[:, 2] - s.labels.boxes[:, 0],
                        s.labels.boxes[:, 3] - s.labels.boxes[:, 1]]), axis=0)
    for s in train
])
sides = sides[np.isfinite(sides)]
print(f"{len(sides)} annotated targets, longer side in source pixels:")
for lo, hi, name in [(0, 8, "tiny"), (8, 16, "small"), (16, 32, "medium"), (32, 10**6, "normal")]:
    print(f"  {name:<7} {lo:>3}-{hi if hi < 10**6 else '+':<4} px   "
          f"{np.mean((sides >= lo) & (sides < hi)):6.1%}")
print(f"  median {np.median(sides):.1f} px, p10 {np.percentile(sides, 10):.1f}, "
      f"p90 {np.percentile(sides, 90):.1f}")

probe_clips = sample_clips(train[:8], length=CLIP_LEN, stride=CLIP_STRIDE, size=SIZE,
                           frame_size=(width, height), jitter=32, seed=SEED)
native = sum(c.native for c in probe_clips)
print(f"\n{len(probe_clips)} clips from the first 8 sequences")
print(f"  {native / len(probe_clips):.1%} on native pixels -- the deployment's crop512")
print(f"  {1 - native / len(probe_clips):.1%} fall back to the resized full frame")

## Mask labels

Anti-UAV410 gives **boxes**; EdgeTAM predicts **masks**. A large SAM 2.1 teacher
runs once, offline, box-prompted per frame, and its masks become the training
target — the same relationship EdgeTAM already has to SAM 2, applied to one
domain.

Two things carry the quality:

- **Zoom.** Prompting the teacher on the full 640×512 frame asks it to segment a
  6-pixel object, and it will not. Prompting on a crop a few times the box size
  is the same model on a much easier problem.
- **Gates.** A mask is kept only if four independent checks agree: the teacher's
  own confidence, agreement with the human box, plausible area, and being one
  connected object. Frames that fail keep their `exist` supervision and fall
  back to a box-projection loss.

The acceptance rate below is a **measurement, not an assumption** — and *which*
gate rejects tells you what to fix.

In [ ]:
from src.training.labels import Gates, Sam2Teacher, label_sequence

GATES   = Gates(teacher_iou=0.7, box_iou=0.6, area=(0.15, 1.3), component=0.8)
teacher = Sam2Teacher(TEACHER_ID, device="cuda")
print(f"teacher {TEACHER_ID} on {torch.cuda.get_device_name(0)}")

In [ ]:
# --- One sequence first, before committing to all of them ---------------
import time
t0 = time.time()
probe = label_sequence(train[0], teacher, LABELS / "train", gates=GATES, zoom=ZOOM,
                       min_size=MIN_CROP, frame_size=(width, height),
                       stride=LABEL_STRIDE, batch_size=TEACHER_BATCH)
elapsed = time.time() - t0

print(probe)
print(f"\n{elapsed:.0f}s for {probe['attempted']} frames "
      f"({1000 * elapsed / max(probe['attempted'], 1):.0f} ms/frame)")
from src.training import frames_to_label
attempts = sum(len(frames_to_label(s, LABEL_STRIDE)) for s in train + val)
print(f"~{attempts * elapsed / max(probe['attempted'], 1) / 60:.0f} min for all "
      f"{len(train) + len(val)} sequences")

assert probe["acceptance_rate"] > 0.3, (
    "fewer than a third of frames produced a usable mask. Raise ZOOM or MIN_CROP, "
    "or loosen the gate named most often in probe['rejected'], before spending "
    "an hour on the rest.")

In [ ]:
# --- Look at them ------------------------------------------------------
import cv2
import matplotlib.pyplot as plt
from src.training import load_window, open_masks

masks = open_masks(LABELS / "train" / train[0].name / "pseudo_masks.npz")
picked = sorted(masks)[::max(len(masks) // 6, 1)][:6]

fig, axes = plt.subplots(1, len(picked), figsize=(3 * len(picked), 3.4))
for ax, idx in zip(np.atleast_1d(axes), picked):
    box = train[0].labels.boxes[idx]
    x0, y0 = max(int(box[0]) - 40, 0), max(int(box[1]) - 40, 0)
    w, h = int(box[2] - box[0]) + 80, int(box[3] - box[1]) + 80
    ax.imshow(load_window(train[0].frames[idx], (x0, y0), (w, h), 128))
    ax.imshow(cv2.resize(masks[idx][y0:y0 + h, x0:x0 + w].astype(np.uint8),
                         (128, 128), interpolation=cv2.INTER_NEAREST),
              alpha=0.45, cmap="autumn")
    ax.set_title(f"frame {idx}"); ax.axis("off")
plt.suptitle(f"{train[0].name}: teacher masks that passed all four gates")
plt.tight_layout(); plt.show()

In [ ]:
# --- The whole subset --------------------------------------------------
# Resumable: a sequence already labelled at this stride is skipped, so a
# dropped Colab connection costs the current sequence and nothing else.
import json
from src.training.labels import REPORT_FILE, summarise
from tqdm.auto import tqdm

def existing(split, sequence):
    path = LABELS / split / sequence.name / REPORT_FILE
    if not path.is_file():
        return None
    report = json.loads(path.read_text())
    ok = report.get("stride") == LABEL_STRIDE and report.get("frames") == len(sequence)
    return report if ok else None

reports, reused = {}, 0
for split, sequences in (("train", train), ("val", val)):
    rows = []
    for sequence in tqdm(sequences, desc=f"labelling {split}"):
        report = existing(split, sequence)
        reused += report is not None
        rows.append(report or label_sequence(
            sequence, teacher, LABELS / split, gates=GATES, zoom=ZOOM,
            min_size=MIN_CROP, frame_size=(width, height),
            stride=LABEL_STRIDE, batch_size=TEACHER_BATCH))
    reports[split] = rows
    print(f"\n### {split}\n{summarise(rows)}\n")
print(f"({reused} sequence(s) reused from an earlier run)")

In [ ]:
# --- Record what was built ---------------------------------------------
manifest = {
    "dataset": "Anti-UAV410", "data_root": str(DATA),
    "frame_size": [width, height], "model_input": SIZE,
    "clip": {"length": CLIP_LEN, "stride": CLIP_STRIDE},
    "teacher": TEACHER_ID, "label_stride": LABEL_STRIDE,
    "zoom": ZOOM, "min_crop": MIN_CROP,
    "gates": {"teacher_iou": GATES.teacher_iou, "box_iou": GATES.box_iou,
              "area": list(GATES.area), "component": GATES.component},
    "sequences": {k: [r["sequence"] for r in v] for k, v in reports.items()},
    "acceptance": {k: sum(r["accepted"] for r in v)
                      / max(sum(r.get("attempted", r["visible"]) for r in v), 1)
                   for k, v in reports.items()},
}
(WORK / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest["acceptance"], indent=2))

In [ ]:
# --- Give the GPU back --------------------------------------------------
# hiera-large is ~2 GB of weights plus its activations. Leaving it resident
# would come straight off the training batch size measured two cells below.
import gc
del teacher
gc.collect(); torch.cuda.empty_cache()
print(f"{torch.cuda.memory_allocated() / 2**30:.2f} GiB allocated, "
      f"{torch.cuda.memory_reserved() / 2**30:.2f} GiB reserved")

## Fine-tuning: partial, not LoRA

| | partial fine-tune | LoRA |
|---|---|---|
| **memory** | EdgeTAM is **13.9 M parameters** (4.92 encoder / 2.96 memory attention / 1.62 memory encoder / 4.41 head). Nothing about training it is memory-bound | solves a problem we do not have |
| **deployment** | weights change, the ONNX graph is identical | a merged adapter *is* just weights; unmerged, it adds ops to the hot path |
| **coverage** | reaches everything | targets `nn.Linear`. The domain shift here is in the **convolutional RepViT trunk**, LoRA's worst case |
| **QAT afterwards** | the same loop with `mtq.quantize` applied | needs real weight updates anyway |

LoRA's one genuine benefit — regularisation on a small single-class dataset — is
bought more cheaply and more precisely by **freezing the right modules**.

### What stays frozen, and why

**The whole memory path**: `memory_attention`, `memory_encoder`,
`spatial_perceiver`, and the learned memory tokens.

1. **It is the write port of a recurrent loop.** This project already measured
   what a systematic change there costs: quantising the memory encoder alone
   gave mean IoU 0.9397, and — unlike every other module — the damage was a
   *sustained decline across the clip* rather than isolated dropouts, because
   its output is what the bank stores and the next seven frames read back
   (`docs/tensorrt_fp16.md`).
2. **Its ONNX rewrite is the intricate one** — fixed memory slots, tiled RoPE
   tables, an additive attention mask. Retraining the weights those graphs were
   derived from invites a silent mismatch between checkpoint and engine.
3. **It operates on abstract features, not pixels.** Thermal-versus-RGB is an
   encoder problem.

### Two more choices worth knowing about

- **The model trains in `eval()` mode.** RepViT is full of batch norms, and
  letting their statistics drift would break the match with the engines
  TensorRT folds them into, and invalidate any INT8 calibration taken before it.
  Independently, SAM 2 withholds `object_score_logits` in training mode.
- **No teacher forcing.** Every frame conditions on the memory *the model
  itself* wrote. Feeding ground truth into the bank would train a model that has
  never seen its own mistakes — which is precisely the failure being fixed.

In [ ]:
# --- The model ----------------------------------------------------------
# image_size_overrides also fixes the cross-attention rotary table, which does
# not self-adjust to a new resolution (src/trackers/_hydra_overrides.py).
from sam2.build_sam import build_sam2_video_predictor
from src.trackers._hydra_overrides import image_size_overrides

model = build_sam2_video_predictor(
    "configs/edgetam.yaml",
    "third_party/EdgeTAM/checkpoints/edgetam.pt",
    device="cuda",
    hydra_overrides_extra=image_size_overrides(SIZE),
)
model.eval()   # deliberate -- see above
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters, "
      f"image_size={model.image_size}")

In [ ]:
from src.training.finetune import Rates, apply_freeze, param_groups, summarise_freeze

print(summarise_freeze(apply_freeze(model, "head"), model))

# Assert the policy rather than trusting the print: a run that quietly trained
# the memory path would only show up as a bad checkpoint hours later.
for name, p in model.named_parameters():
    if name.startswith(("memory_attention", "memory_encoder", "spatial_perceiver")):
        assert not p.requires_grad, name

In [ ]:
# --- Clips, paired with their pseudo-masks ------------------------------
# open_masks holds the run-length encoding and decodes a frame when the loader
# asks for it. load_masks would materialise every mask up front: a 512x640
# boolean frame is 328 KB, and this subset has tens of thousands of them.
from src.training import open_masks

def build(split, sequences, jitter):
    stores = {s.name: open_masks(LABELS / split / s.name / "pseudo_masks.npz")
              for s in sequences}
    clips = sample_clips(sequences, length=CLIP_LEN, stride=CLIP_STRIDE, size=SIZE,
                         frame_size=(width, height), jitter=jitter, seed=SEED)
    return clips, stores

train_clips, train_stores = build("train", train, jitter=32)
val_clips,   val_stores   = build("val", val, jitter=0)
labelled = sum(len(s) for s in train_stores.values())
print(f"train {len(train_clips)} clips / val {len(val_clips)} clips of {CLIP_LEN} frames")
print(f"{labelled} labelled frames across {len(train_stores)} train sequences; "
      f"the rest train on `exist` + box projection")

## First: overfit one clip

Before spending GPU hours, prove the loop can learn *anything*. One clip, no
augmentation, frame 0 included in the loss — this should collapse towards zero
within a couple of hundred steps.

If it does not, the problem is in the plumbing (prompt coordinates, mask
alignment, the memory bookkeeping), and no amount of real training will fix it.
This is the cheapest possible place to find that out.

In [ ]:
import copy
from src.training.clip_loop import clip_losses, collate

# The clip with the most of *its own* frames labelled -- a sequence-level count
# would happily pick a clip whose eight frames all fell in the labelling stride.
def labelled_frames(clip):
    store = train_stores[clip.sequence.name]
    return sum(int(i) in store for i in clip.indices)

probe_clip = max(train_clips[:4000], key=labelled_frames)
probe = collate([probe_clip], [train_stores[probe_clip.sequence.name]], "cpu").to("cuda")
print(f"{probe_clip.sequence.name} frames {probe_clip.indices}, "
      f"{labelled_frames(probe_clip)} of {CLIP_LEN} with a teacher mask")

snapshot = copy.deepcopy(model.state_dict())
opt = torch.optim.AdamW(param_groups(model, Rates(head=3e-4)))
trainable = [p for g in opt.param_groups for p in g["params"]]
history = []
for step in range(120):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss, terms = clip_losses(model, probe, skip_first=False)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable, 1.0)
    opt.step()
    history.append(float(loss))
    if step % 20 == 0:
        print(f"step {step:>4}  loss {float(loss):7.4f}   " +
              "  ".join(f"{k} {v:.3f}" for k, v in terms.items()))

drop = history[0] / max(min(history[-10:]), 1e-6)
print(f"\nloss fell {drop:.1f}x  ({history[0]:.3f} -> {min(history[-10:]):.3f})")
assert drop > 3.0, ("the loop cannot even overfit one clip. Check prompt "
                    "coordinates, mask alignment and the memory bookkeeping "
                    "before training on anything larger.")
model.load_state_dict(snapshot)          # throw the overfit away
del snapshot, opt, probe
gc.collect(); torch.cuda.empty_cache()

## How many clips at once

**The batch dimension is clips, not frames.** SAM 2 batches *objects*, and each
row of that batch carries its own memory, its own object pointer and its own
object score — the attention never mixes rows. So N independent clips ride the
same machinery N tracked objects would, with no change to the model.

The largest N is a property of the card, not of the recipe, and it is measured
rather than guessed: a real forward *and backward* at each candidate size —
the backward is where the activation graph is actually held — and the largest
one that leaves 15 % of the card free for fragmentation, the EMA copy and the
validation pass.

The other half of using a big batch is **keeping it fed**. One batch of 16 clips
is 128 JPEGs to decode, crop and normalise: ~0.4 s of OpenCV against ~0.15 s of
compute, so on the training thread the GPU would idle two thirds of every step
and a bigger card would buy nothing. `prefetch` moves that onto worker threads —
`cv2.imread` releases the GIL, so it is real parallelism — and hands the loop
batches that are already assembled.

`LOADER_WORKERS` and `PREFETCH_DEPTH` pull in different directions and are
separate for that reason. Workers is how many clips are read at once and wants
to be around the core count; depth is how many *batches* sit in host RAM ahead
of the GPU and wants to stay at two. At `BATCH = 64` one batch is 1.6 GB, so a
queue length tied to the thread count would be tens of gigabytes of decoded
JPEG waiting on a card that is content with two.

In [ ]:
from src.training.loader import auto_batch_size, batch_clips, prefetch

# Measure under the *encoder* stage's freeze, which is the expensive one: with
# only the head trainable the trunk's activations are never kept for a
# backward, so a batch sized there would OOM the moment the encoder unfreezes.
apply_freeze(model, "encoder")
BATCH = auto_batch_size(model, train_clips, train_stores, maximum=BATCH_CEILING)
apply_freeze(model, "head")
ACCUM = 1                       # the measured batch is already large; no need

print(f"\ntraining on {BATCH} clips x {CLIP_LEN} frames = "
      f"{BATCH * CLIP_LEN} frames per step")

In [ ]:
# --- Is the loader actually keeping up? ---------------------------------
# Loading and compute are timed apart, one batch held at a time -- keeping four
# 512x512x8-frame batches alive to time them would itself cost gigabytes.
# If loading wins, raise LOADER_WORKERS; if compute wins, the GPU is the limit
# and the batch above is the right one.
import time

def batch_stream(n):
    return prefetch(batch_clips(train_clips, BATCH, seed=SEED, limit=n),
                    train_stores, "cuda", workers=LOADER_WORKERS,
                    depth=PREFETCH_DEPTH)

stream = batch_stream(6)
held = next(stream)                            # warm the workers and the caches
t0 = time.time()
for batch in stream:                           # consumed and dropped immediately
    del batch
load = (time.time() - t0) / 5

t0 = time.time()
for _ in range(4):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        clip_losses(model, held)[0].backward()
    model.zero_grad(set_to_none=True)
torch.cuda.synchronize()
step = (time.time() - t0) / 4

print(f"loading {load * 1000:6.0f} ms/batch (behind {LOADER_WORKERS} workers)")
print(f"compute {step * 1000:6.0f} ms/batch")
print(f"-> {'GPU-bound, good' if step > load else 'input-bound: raise LOADER_WORKERS'}")
del held; gc.collect(); torch.cuda.empty_cache()

## Training

Two stages. The head first, alone, so the mask decoder and the object-score head
adapt to thermal statistics before the features under them start moving; then
the encoder joins at a tenth of the rate.

An "epoch" here is `STEPS_PER_EPOCH` batches of a reshuffled pass, not the whole
clip pool — every frame is a clip start, so consecutive clips share seven of
their eight frames and a true pass over ~60 000 of them is a day of near
duplicates. The EMA is rebuilt per stage, because the set of trainable
parameters is what it averages and that set changes when the encoder unfreezes.

In [ ]:
from src.training.finetune import EMA, save_checkpoint

STAGES = [("head",    1, Rates(head=1e-4)),
          ("encoder", 2, Rates(head=5e-5, neck=5e-5, trunk=1e-5))]

# Mean clip loss over a fixed slice of val -- the same clips every time, so
# epoch-to-epoch differences are the model rather than the sample.
@torch.no_grad()
def validate(batches=VAL_BATCHES):
    chunks = batch_clips(val_clips, BATCH, seed=1, limit=batches)
    losses = []
    for batch in prefetch(chunks, val_stores, "cuda", workers=LOADER_WORKERS,
                          depth=PREFETCH_DEPTH):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            losses.append(float(clip_losses(model, batch)[0]))
    return float(np.mean(losses)) if losses else float("nan")

best, log = float("inf"), []
for stage, epochs, rates in STAGES:
    print(f"\n===== stage {stage!r}: {epochs} epoch(s) of {STEPS_PER_EPOCH} steps =====")
    print(summarise_freeze(apply_freeze(model, stage), model))
    opt = torch.optim.AdamW(param_groups(model, rates))
    trainable = [p for g in opt.param_groups for p in g["params"]]
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[g["lr"] for g in opt.param_groups],
        total_steps=max(epochs * STEPS_PER_EPOCH // ACCUM, 1), pct_start=0.1)
    ema = EMA(model, decay=0.999)

    for epoch in range(epochs):
        chunks = batch_clips(train_clips, BATCH, seed=SEED + 100 * epoch,
                             limit=STEPS_PER_EPOCH)
        bar = tqdm(prefetch(chunks, train_stores, "cuda", workers=LOADER_WORKERS,
                            depth=PREFETCH_DEPTH),
                   total=STEPS_PER_EPOCH, desc=f"{stage} e{epoch}")
        for step, batch in enumerate(bar):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss, terms = clip_losses(model, batch)
            (loss / ACCUM).backward()
            if (step + 1) % ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                opt.step(); opt.zero_grad(set_to_none=True); sched.step()
                ema.update(model)
            bar.set_postfix(loss=f"{float(loss):.3f}",
                            **{k: f"{v:.2f}" for k, v in terms.items()})

        with ema.applied(model):
            score = validate()
            marker = ""
            if score < best:
                best, marker = score, "  <- saved"
                save_checkpoint(model, CKPT / "edgetam_thermal_512.pt",
                                {"stage": stage, "epoch": epoch, "val_loss": score,
                                 "image_size": SIZE, "dataset": "Anti-UAV410",
                                 "batch": BATCH, "teacher": TEACHER_ID})
        log.append({"stage": stage, "epoch": epoch, "val": score})
        print(f"  epoch {epoch}: val clip loss {score:.4f}{marker}")

    del opt, sched, ema
    gc.collect(); torch.cuda.empty_cache()

print(f"\nbest val clip loss {best:.4f} -> {CKPT / 'edgetam_thermal_512.pt'}")

In [ ]:
# --- Free the training graph before the evaluation runs -----------------
# tools/eval_antiuav.py starts its own tracker in a subprocess; this model's
# weights are already on disk and its optimiser state is worth nothing now.
del model, train_clips, val_clips
gc.collect(); torch.cuda.empty_cache()
print(f"{torch.cuda.memory_reserved() / 2**30:.2f} GiB reserved")

## Did it actually help?

Clip loss is a proxy. The number that decides anything is **state accuracy on
held-out sequences, measured through the deployment path** — the same
`VideoTracker` the Orin runs, prompted once on the first annotated frame and
left to propagate.

`tools/eval_antiuav.py` also reports **dropout episodes**: how often a visible
target was lost and for how long. That is the statistic the mean hides, and it
is the one that describes this project's actual failure — one hard frame writes
`no_obj_ptr` into the memory bank and the next frames read it back. If the
`exist` supervision worked, the episodes get *shorter*, not just rarer.

Same resolution, same tracker, same sequences — only the weights differ.

In [ ]:
!python tools/eval_antiuav.py --data {DATA} --split val --limit {VAL_SEQUENCES} \
    --tracker edgetam --config configs/edgetam_512.yaml --mode crop \
    --json /content/eval_val_stock.json 2>&1 | tail -8

In [ ]:
!python tools/eval_antiuav.py --data {DATA} --split val --limit {VAL_SEQUENCES} \
    --tracker edgetam --config configs/edgetam_512_thermal.yaml --mode crop \
    --json /content/eval_val_thermal.json 2>&1 | tail -8

In [ ]:
import matplotlib.pyplot as plt

def load(path):
    return json.loads(Path(path).read_text())["sequences"]

def weighted(rows, key):
    frames = sum(r["frames"] for r in rows)
    return sum(r[key] * r["frames"] for r in rows) / max(frames, 1)

def compare(stock, tuned, title):
    print(f"{title}\n{'':<12}{'state acc':>12}{'success AUC':>14}{'lost frames':>14}"
          f"{'episodes':>11}")
    for label, rows in (("stock", stock), ("fine-tuned", tuned)):
        lost = sum(sum(r["dropout_lengths"]) for r in rows)
        episodes = sum(len(r["dropout_lengths"]) for r in rows)
        print(f"{label:<12}{weighted(rows, 'state_accuracy'):>12.4f}"
              f"{weighted(rows, 'success_auc'):>14.4f}{lost:>14}{episodes:>11}")

stock, tuned = load("/content/eval_val_stock.json"), load("/content/eval_val_thermal.json")
compare(stock, tuned, "val")

by_name = {s["name"]: s for s in tuned}
names = [s["name"] for s in stock if s["name"] in by_name]
fig, ax = plt.subplots(figsize=(7, 0.3 * len(names) + 1.5))
ax.barh(range(len(names)),
        [by_name[n]["state_accuracy"] - next(s for s in stock if s["name"] == n)["state_accuracy"]
         for n in names])
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("state accuracy: fine-tuned − stock  (val)")
plt.tight_layout(); plt.show()

### And once, on `test`

`val` chose the checkpoint, so `val` cannot also be the report. This is the only
cell that touches the test split, and nothing after it changes anything.

In [ ]:
assert "test" in splits, ("the test split was not extracted -- add it to SPLITS "
                          "and re-run the fetch cell")
!python tools/eval_antiuav.py --data {DATA} --split test --limit {TEST_SEQUENCES} \
    --tracker edgetam --config configs/edgetam_512.yaml --mode crop \
    --json /content/eval_test_stock.json 2>&1 | tail -5
!python tools/eval_antiuav.py --data {DATA} --split test --limit {TEST_SEQUENCES} \
    --tracker edgetam --config configs/edgetam_512_thermal.yaml --mode crop \
    --json /content/eval_test_thermal.json 2>&1 | tail -5

In [ ]:
compare(load("/content/eval_test_stock.json"),
        load("/content/eval_test_thermal.json"), "test (held out)")

In [ ]:
# --- Keep it ------------------------------------------------------------
import shutil

summary = {"stages": [s[0] for s in STAGES], "batch": BATCH, "clip": CLIP_LEN,
           "steps_per_epoch": STEPS_PER_EPOCH, "best_val_clip_loss": best,
           "log": log, "manifest": manifest}
(WORK / "finetune_log.json").write_text(json.dumps(summary, indent=2) + "\n")

if MIRROR is not None:
    shutil.copy(CKPT / "edgetam_thermal_512.pt", MIRROR / "edgetam_thermal_512.pt")
    shutil.copy(WORK / "finetune_log.json", MIRROR / "finetune_log.json")
    shutil.copy(WORK / "manifest.json", MIRROR / "manifest.json")
    for split in ("train", "val"):        # ~1 MB: RLE, not bitmaps
        shutil.copytree(LABELS / split, MIRROR / "labels" / split, dirs_exist_ok=True)
    print(f"checkpoint + labels -> {MIRROR}")
else:
    print(f"checkpoint at {CKPT / 'edgetam_thermal_512.pt'} -- download it before "
          f"this runtime is recycled")

## What you have, and what to check

`checkpoints/edgetam_thermal_512.pt` — in EdgeTAM's own `{"model": state_dict}`
layout, so `configs/edgetam_512_thermal.yaml` already points at it and the
export/build chain needs no change:

```bash
python tools/export_edgetam_onnx.py --outdir models512_ft/ --image-size 512 \
    --checkpoint checkpoints/edgetam_thermal_512.pt --verify
python tools/build_trt_engines.py --outdir models512_ft/ --max-batch 4
```

**Read the comparison honestly.** Four outcomes and what each one means:

| what you see | what it means |
|---|---|
| state accuracy up, dropout episodes *shorter* | the `exist` supervision landed — this is the win being sought |
| state accuracy up, dropouts unchanged | masks got better, the object head did not. Raise `Weights.object_score` |
| state accuracy up on `val`, flat on `test` | the subset was too small, or `STEPS_PER_EPOCH` too many for it. Raise `TRAIN_SEQUENCES` before anything else |
| down on some sequences | check whether those are the `full512` clips: a target that outruns a fixed window trains on a distorted resize. `manifest["clip"]` and `Clip.native` tell you |

**Next:** `03_quantization_v2_int8.ipynb`. It calibrates on *this* checkpoint —
INT8 scales taken from the stock model would be calibrated for the wrong
activation distributions.